In [ ]:
import os
import json
import asyncio
from typing import TypedDict, List
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_postgres import PGVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END

# --- 환경 설정 ---
load_dotenv()
print("=" * 80)
print("LangGraph를 사용한 Corrective RAG (CRAG) 시스템")
print("=" * 80)


# =================================================================
#  Part 1: 데이터 준비 및 Retriever/Tool 생성
# =================================================================
print("\n[Part 1] 데이터 준비 및 도구 생성")
print("-" * 80)

# 1. 문서 로드 및 분할 (예시 PDF 사용)
try:
    FILE_PATH = "13. LangGraph/data/SPRi AI Brief_10월호_산업동향_1002_F.pdf"
    loader = PyPDFLoader(FILE_PATH)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(documents)
    print(f"'{os.path.basename(FILE_PATH)}' 문서를 {len(splits)}개의 청크로 분할 완료.")

    # 2. 임베딩 및 Vector DB 저장
    embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # PostgreSQL 연결 설정
    db_config = {
        'host': 'localhost',
        'port': 5432,
        'dbname': 'testdb',
        'user': 'test',
        'password': '5748'
    }
    CONNECTION_STRING = f"postgresql+psycopg://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['dbname']}"
    COLLECTION_NAME = "crag_example"

    vectorstore = PGVector.from_documents(
        documents=splits,
        embedding=embeddings_model,
        collection_name=COLLECTION_NAME,
        connection=CONNECTION_STRING,
        pre_delete_collection=True,
    )
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    print(f"Vector DB에 데이터 저장 및 Retriever 생성 완료.")

except Exception as e:
    print(f"[오류] 데이터 준비 중 문제가 발생했습니다: {e}")
    print("PDF 파일 경로와 PostgreSQL DB 연결 정보를 확인해주세요.")
    exit()

# 3. Web Search 도구 정의
@tool
def web_search(query: str):
    """웹 검색을 수행하여 최신 정보를 얻습니다."""
    search = GoogleSerperAPIWrapper()
    results = search.run(query)
    return results

# ============================================================
#  Part 2: LangGraph 설계 (상태, 노드, 엣지 정의)
# ============================================================
print("\n[Part 2] LangGraph 설계")
print("-" * 80)

# --- 2-1. 그래프 상태 (State) 정의 ---
class GraphState(TypedDict):
    question: str
    documents: List[Document]
    generation: str
    decision: str # 문서 평가 결과 ('yes', 'no', 'web_search')

# --- 2-2. 노드(Node) 함수 정의 ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Pydantic 모델을 사용하여 문서 평가 결과의 스키마를 정의
class Grader(BaseModel):
    """문서의 관련성을 평가하기 위한 이진 점수입니다."""
    decision: str = Field(description="결정은 'yes', 'no', 'web_search' 중 하나여야 합니다.")

def retrieve(state: GraphState):
    """Vector DB에서 문서를 검색합니다."""
    print("--- 노드 실행: retrieve ---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents}

def grade_documents(state: GraphState):
    """검색된 문서가 질문에 답변하기에 충분한지, 웹 검색이 필요한지, 관련이 없는지 평가합니다."""
    print("--- 노드 실행: grade_documents ---")
    question = state["question"]
    documents = state["documents"]

    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 문서 평가 전문가입니다. 주어진 문서가 사용자의 질문에 답변하기에 충분한지, 아니면 웹 검색이 추가로 필요한지, 또는 전혀 관련이 없는지 평가해주세요.
        다음 세 가지 중 하나로만 답변해야 합니다: 'yes', 'no', 'web_search'.

        - 'yes': 문서만으로 질문에 완벽하게 답변할 수 있습니다.
        - 'web_search': 문서가 관련은 있지만, 최신 정보나 추가적인 내용 보강을 위해 웹 검색이 필요합니다.
        - 'no': 문서가 질문과 전혀 관련이 없습니다.

        질문: {question}
        
        문서 내용:
        {documents}
        """),
    ])
    
    # Pydantic 모델을 사용하여 LLM 출력 구조를 지정
    structured_llm_grader = llm.with_structured_output(Grader)
    
    doc_str = "\n\n".join(doc.page_content for doc in documents)
    chain = prompt | structured_llm_grader
    response = chain.invoke({"question": question, "documents": doc_str})
    
    print(f"   문서 평가 결과: {response.decision}")
    return {"decision": response.decision}

def generate(state: GraphState):
    """검색된 정보를 바탕으로 답변을 생성합니다."""
    print("--- 노드 실행: generate ---")
    question = state["question"]
    documents = state["documents"]
    
    prompt = ChatPromptTemplate.from_template("""
    주어진 문맥 정보를 사용하여 다음 질문에 답변해주세요.
    
    질문: {question}
    
    문맥:
    {context}
    """)
    
    context = "\n\n".join(f"[출처: {doc.metadata.get('source', '문서')}]\n{doc.page_content}" for doc in documents)
    chain = prompt | llm | StrOutputParser()
    generation = chain.invoke({"context": context, "question": question})
    
    return {"generation": generation}

def transform_query(state: GraphState):
    """웹 검색에 더 적합하도록 질문을 재구성합니다."""
    print("--- 노드 실행: transform_query ---")
    question = state["question"]
    
    prompt = ChatPromptTemplate.from_template("""
    당신은 쿼리 생성 전문가입니다. 사용자의 질문을 웹 검색에 더 효과적인 검색어로 재구성해주세요.
    재구성된 검색어만 반환해야 합니다.

    원래 질문: {question}
    """)
    
    chain = prompt | llm | StrOutputParser()
    better_question = chain.invoke({"question": question})
    
    print(f"   재구성된 질문: {better_question}")
    return {"question": better_question}

def web_search_node(state: GraphState):
    """재구성된 질문으로 웹 검색을 수행합니다. (기존 문서는 대체됩니다)"""
    print("--- 노드 실행: web_search_node ---")
    question = state["question"]
    search_result = web_search.invoke(question)
    web_documents = [Document(page_content=search_result, metadata={"source": "web_search"})]
    return {"documents": web_documents}

def web_search_for_supplement(state: GraphState):
    """보충을 위해 웹 검색을 수행하고 기존 문서에 결과를 추가합니다."""
    print("--- 노드 실행: web_search_for_supplement ---")
    question = state["question"]
    existing_documents = state["documents"]
    
    search_result = web_search.invoke(question)
    web_documents = [Document(page_content=search_result, metadata={"source": "web_search"})]
    
    combined_documents = existing_documents + web_documents
    print(f"   기존 문서 {len(existing_documents)}개와 웹 검색 결과 {len(web_documents)}개를 결합.")
    return {"documents": combined_documents}

# --- 2-3. 조건부 엣지(Edge) 함수 정의 ---
def route_after_grading(state: GraphState):
    """문서 평가 결과에 따라 다음 노드를 결정합니다."""
    print("--- 조건부 엣지 실행: route_after_grading ---")
    decision = state["decision"]
    if decision == "yes":
        print("   (결정) 문서 충분 -> generate")
        return "generate"
    elif decision == "no":
        print("   (결정) 문서 관련 없음 -> transform_query (교정)")
        return "transform_query"
    elif decision == "web_search":
        print("   (결정) 문서 보충 필요 -> web_search_for_supplement")
        return "web_search"

# --- 2-4. 그래프 생성 및 연결 ---
workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_search_node", web_search_node)
workflow.add_node("web_search_for_supplement", web_search_for_supplement) # 새 노드

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", route_after_grading, {
    "generate": "generate",
    "transform_query": "transform_query",
    "web_search": "web_search_for_supplement", # 새 경로
})
workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "generate")
workflow.add_edge("web_search_for_supplement", "generate") # 새 경로
workflow.add_edge("generate", END)

app = workflow.compile()
print("\nCRAG 그래프 컴파일 완료!")
try:
    img_bytes = app.get_graph().draw_mermaid_png()
    with open("crag_graph.png", "wb") as f:
        f.write(img_bytes)
    print("그래프 구조를 'crag_graph.png' 파일로 저장했습니다.")
except Exception as e:
    print(f"그래프 시각화 실패 (Graphviz 필요): {e}")

# ============================================================
#  Part 3: CRAG 시스템 실행 (비동기 스트리밍)
# ============================================================
async def main():
    print("\n" + "=" * 80)
    print("Corrective RAG 시스템을 시작합니다.")
    print("종료하려면 'quit' 또는 'exit'를 입력하세요.")
    print("=" * 80)

    while True:
        user_question = input("\nYou: ").strip()
        if user_question.lower() in ['quit', 'exit', '종료']:
            print("\n시스템을 종료합니다.")
            break
        if not user_question:
            continue

        inputs = {"question": user_question}
        try:
            print("\nAI: ", end="", flush=True)
            currently_generating = False
            async for event in app.astream_events(inputs, version="v2"):
                kind = event["event"]
                
                # 'generate' 노드가 실행될 때 스트리밍 시작 플래그 설정
                if kind == "on_chain_start" and event["name"] == "generate":
                    currently_generating = True
                
                # 'generate' 노드가 끝나면 스트리밍 중지 플래그 설정
                elif kind == "on_chain_end" and event["name"] == "generate":
                    currently_generating = False

                # 스트리밍 중일 때 LLM이 생성하는 텍스트 청크를 실시간으로 출력
                elif kind == "on_chat_model_stream" and currently_generating:
                    content = event["data"]["chunk"].content
                    if content:
                        print(content, end="", flush=True)
            
            print("\n" + "-" * 80)

        except Exception as e:
            print(f"\n[오류 발생] {e}")

if __name__ == "__main__":
    asyncio.run(main())



In [ ]:
import os
import json
import asyncio
from typing import TypedDict, List
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_postgres import PGVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END

# --- 환경 설정 ---
load_dotenv()
print("=" * 80)
print("LangGraph를 사용한 Corrective RAG (CRAG) 시스템")
print("=" * 80)


# =================================================================
#  Part 1: 데이터 준비 및 Retriever/Tool 생성
# =================================================================
print("\n[Part 1] 데이터 준비 및 도구 생성")
print("-" * 80)

# 1. 문서 로드 및 분할 (예시 PDF 사용)
try:
    FILE_PATH = "13. LangGraph/data/SPRi AI Brief_10월호_산업동향_1002_F.pdf"
    loader = PyPDFLoader(FILE_PATH)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(documents)
    print(f"'{os.path.basename(FILE_PATH)}' 문서를 {len(splits)}개의 청크로 분할 완료.")

    # 2. 임베딩 및 Vector DB 저장
    embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # PostgreSQL 연결 설정
    db_config = {
        'host': 'localhost',
        'port': 5432,
        'dbname': 'testdb',
        'user': 'test',
        'password': '5748'
    }
    CONNECTION_STRING = f"postgresql+psycopg://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['dbname']}"
    COLLECTION_NAME = "crag_example"

    vectorstore = PGVector.from_documents(
        documents=splits,
        embedding=embeddings_model,
        collection_name=COLLECTION_NAME,
        connection=CONNECTION_STRING,
        pre_delete_collection=True,
    )
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    print(f"Vector DB에 데이터 저장 및 Retriever 생성 완료.")

except Exception as e:
    print(f"[오류] 데이터 준비 중 문제가 발생했습니다: {e}")
    print("PDF 파일 경로와 PostgreSQL DB 연결 정보를 확인해주세요.")
    exit()

# 3. Web Search 도구 정의
@tool
def web_search(query: str):
    """웹 검색을 수행하여 최신 정보를 얻습니다."""
    search = GoogleSerperAPIWrapper()
    results = search.run(query)
    return results

# ============================================================
#  Part 2: LangGraph 설계 (상태, 노드, 엣지 정의)
# ============================================================
print("\n[Part 2] LangGraph 설계")
print("-" * 80)

# --- 2-1. 그래프 상태 (State) 정의 ---
class GraphState(TypedDict):
    question: str
    documents: List[Document]
    generation: str
    decision: str # 문서 평가 결과 ('yes', 'no', 'web_search')

# --- 2-2. 노드(Node) 함수 정의 ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Pydantic 모델을 사용하여 문서 평가 결과의 스키마를 정의
class Grader(BaseModel):
    """문서의 관련성을 평가하기 위한 이진 점수입니다."""
    decision: str = Field(description="결정은 'yes', 'no', 'web_search' 중 하나여야 합니다.")

def retrieve(state: GraphState):
    """Vector DB에서 문서를 검색합니다."""
    print("--- 노드 실행: retrieve ---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents}

def grade_documents(state: GraphState):
    """검색된 문서가 질문에 답변하기에 충분한지, 웹 검색이 필요한지, 관련이 없는지 평가합니다."""
    print("--- 노드 실행: grade_documents ---")
    question = state["question"]
    documents = state["documents"]

    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 문서 평가 전문가입니다. 주어진 문서가 사용자의 질문에 답변하기에 충분한지, 아니면 웹 검색이 추가로 필요한지, 또는 전혀 관련이 없는지 평가해주세요.
        다음 세 가지 중 하나로만 답변해야 합니다: 'yes', 'no', 'web_search'.

        - 'yes': 문서만으로 질문에 완벽하게 답변할 수 있습니다.
        - 'web_search': 문서가 관련은 있지만, 최신 정보나 추가적인 내용 보강을 위해 웹 검색이 필요합니다.
        - 'no': 문서가 질문과 전혀 관련이 없습니다.

        질문: {question}
        
        문서 내용:
        {documents}
        """),
    ])
    
    # Pydantic 모델을 사용하여 LLM 출력 구조를 지정
    structured_llm_grader = llm.with_structured_output(Grader)
    
    doc_str = "\n\n".join(doc.page_content for doc in documents)
    chain = prompt | structured_llm_grader
    response = chain.invoke({"question": question, "documents": doc_str})
    
    print(f"   문서 평가 결과: {response.decision}")
    return {"decision": response.decision}

def generate(state: GraphState):
    """검색된 정보를 바탕으로 답변을 생성합니다."""
    print("--- 노드 실행: generate ---")
    question = state["question"]
    documents = state["documents"]
    
    prompt = ChatPromptTemplate.from_template("""
    주어진 문맥 정보를 사용하여 다음 질문에 답변해주세요.
    
    질문: {question}
    
    문맥:
    {context}
    """)
    
    context = "\n\n".join(f"[출처: {doc.metadata.get('source', '문서')}]\n{doc.page_content}" for doc in documents)
    chain = prompt | llm | StrOutputParser()
    generation = chain.invoke({"context": context, "question": question})
    
    return {"generation": generation}

def transform_query(state: GraphState):
    """웹 검색에 더 적합하도록 질문을 재구성합니다."""
    print("--- 노드 실행: transform_query ---")
    question = state["question"]
    
    prompt = ChatPromptTemplate.from_template("""
    당신은 쿼리 생성 전문가입니다. 사용자의 질문을 웹 검색에 더 효과적인 검색어로 재구성해주세요.
    재구성된 검색어만 반환해야 합니다.

    원래 질문: {question}
    """)
    
    chain = prompt | llm | StrOutputParser()
    better_question = chain.invoke({"question": question})
    
    print(f"   재구성된 질문: {better_question}")
    return {"question": better_question}

def web_search_node(state: GraphState):
    """재구성된 질문으로 웹 검색을 수행합니다. (기존 문서는 대체됩니다)"""
    print("--- 노드 실행: web_search_node ---")
    question = state["question"]
    search_result = web_search.invoke(question)
    web_documents = [Document(page_content=search_result, metadata={"source": "web_search"})]
    return {"documents": web_documents}

def web_search_for_supplement(state: GraphState):
    """보충을 위해 웹 검색을 수행하고 기존 문서에 결과를 추가합니다."""
    print("--- 노드 실행: web_search_for_supplement ---")
    question = state["question"]
    existing_documents = state["documents"]
    
    search_result = web_search.invoke(question)
    web_documents = [Document(page_content=search_result, metadata={"source": "web_search"})]
    
    combined_documents = existing_documents + web_documents
    print(f"   기존 문서 {len(existing_documents)}개와 웹 검색 결과 {len(web_documents)}개를 결합.")
    return {"documents": combined_documents}

# --- 2-3. 조건부 엣지(Edge) 함수 정의 ---
def route_after_grading(state: GraphState):
    """문서 평가 결과에 따라 다음 노드를 결정합니다."""
    print("--- 조건부 엣지 실행: route_after_grading ---")
    decision = state["decision"]
    if decision == "yes":
        print("   (결정) 문서 충분 -> generate")
        return "generate"
    elif decision == "no":
        print("   (결정) 문서 관련 없음 -> transform_query (교정)")
        return "transform_query"
    elif decision == "web_search":
        print("   (결정) 문서 보충 필요 -> web_search_for_supplement")
        return "web_search"

# --- 2-4. 그래프 생성 및 연결 ---
workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_search_node", web_search_node)
workflow.add_node("web_search_for_supplement", web_search_for_supplement) # 새 노드

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", route_after_grading, {
    "generate": "generate",
    "transform_query": "transform_query",
    "web_search": "web_search_for_supplement", # 새 경로
})
workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "generate")
workflow.add_edge("web_search_for_supplement", "generate") # 새 경로
workflow.add_edge("generate", END)

app = workflow.compile()
print("\nCRAG 그래프 컴파일 완료!")
try:
    img_bytes = app.get_graph().draw_mermaid_png()
    with open("crag_graph.png", "wb") as f:
        f.write(img_bytes)
    print("그래프 구조를 'crag_graph.png' 파일로 저장했습니다.")
except Exception as e:
    print(f"그래프 시각화 실패 (Graphviz 필요): {e}")

# ============================================================
#  Part 3: CRAG 시스템 실행 (비동기 스트리밍)
# ============================================================
async def main():
    print("\n" + "=" * 80)
    print("Corrective RAG 시스템을 시작합니다.")
    print("종료하려면 'quit' 또는 'exit'를 입력하세요.")
    print("=" * 80)

    while True:
        user_question = input("\nYou: ").strip()
        if user_question.lower() in ['quit', 'exit', '종료']:
            print("\n시스템을 종료합니다.")
            break
        if not user_question:
            continue

        inputs = {"question": user_question}
        try:
            print("\nAI: ", end="", flush=True)
            currently_generating = False
            async for event in app.astream_events(inputs, version="v2"):
                kind = event["event"]
                
                # 'generate' 노드가 실행될 때 스트리밍 시작 플래그 설정
                if kind == "on_chain_start" and event["name"] == "generate":
                    currently_generating = True
                
                # 'generate' 노드가 끝나면 스트리밍 중지 플래그 설정
                elif kind == "on_chain_end" and event["name"] == "generate":
                    currently_generating = False

                # 스트리밍 중일 때 LLM이 생성하는 텍스트 청크를 실시간으로 출력
                elif kind == "on_chat_model_stream" and currently_generating:
                    content = event["data"]["chunk"].content
                    if content:
                        print(content, end="", flush=True)
            
            print("\n" + "-" * 80)

        except Exception as e:
            print(f"\n[오류 발생] {e}")

if __name__ == "__main__":
    asyncio.run(main())



#### langgraph_agent_example_Hierarchical_Multi_Agent

In [ ]:
import os
import asyncio
from typing import TypedDict, List, Annotated, Dict, Union
import operator
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode

# --- 환경 설정 ---
load_dotenv()
print("=" * 80); print("LangGraph를 사용한 심층 계층적 에이전트 팀 시스템"); print("=" * 80)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# =================================================================
#  Part 1: 도구 정의
# =================================================================
print("\n[Part 1] 도구 정의")
print("-" * 80)


@tool
def web_search(query: str) -> str:
    """웹에서 정보를 검색합니다."""
    print(f"--- TOOL: web_search (query: {query}) ---")
    search = GoogleSerperAPIWrapper()
    return search.run(query)

@tool
def write_document(topic: str, notes: str) -> str:
    """주어진 주제와 노트를 바탕으로 문서를 작성합니다."""
    print(f"--- TOOL: write_document (topic: {topic}) ---")
    prompt = ChatPromptTemplate.from_template("다음 주제와 노트에 대한 문서를 작성해주세요.\n주제: {topic}\n노트:\n{notes}")
    return (prompt | llm | StrOutputParser()).invoke({"topic": topic, "notes": notes})

@tool
def take_notes(raw_text: str) -> str:
    """방대한 텍스트를 핵심만 요약하여 노트로 만듭니다."""
    print("--- TOOL: take_notes ---")
    prompt = ChatPromptTemplate.from_template("다음 텍스트를 간결한 노트 형식으로 요약해주세요.\n\n{text}")
    return (prompt | llm | StrOutputParser()).invoke({"text": raw_text})

@tool
def generate_chart(topic: str, data: str) -> str:
    """데이터를 바탕으로 차트에 대한 텍스트 설명을 생성합니다."""
    print(f"--- TOOL: generate_chart (topic: {topic}) ---")
    # 실제 차트 라이브러리를 연동 X, 여기서는 텍스트 설명으로 대체
    return f"[Chart generated for '{topic}' based on data: '{data[:50]}...']"

# =================================================================
#  Part 2: Worker Agent Layer (실무자 계층)
# =================================================================

# TypedDict: 딕셔너리의 키와 값의 타입을 명시하는 타입 힌트
# Annotated: 타입에 추가 메타데이터를 붙이기 위한 도구
# operator.add: 리스트를 병합할 때 사용하는 연산자 (messages를 누적할 때 사용)
class WorkerState(TypedDict): 
    # messages 필드는 BaseMessage의 리스트이며, 
    # operator.add를 통해 새 메시지가 기존 리스트에 추가(누적)됩니다
    # 예: 기존 [msg1, msg2] + 새로운 [msg3] = [msg1, msg2, msg3]
    messages: Annotated[List[BaseMessage], operator.add]

def create_worker_agent(name: str, tools: List, system_prompt: str):
    """
    Worker Agent를 생성하는 팩토리 함수
    
    매개변수
        name: 에이전트의 이름 (디버깅/로깅용)
        tools: 이 에이전트가 사용할 수 있는 도구들의 리스트
        system_prompt: 에이전트의 역할과 행동을 정의하는 시스템 프롬프트
    
    반환값
        컴파일된 LangGraph 워크플로우 (실행 가능한 그래프 객체)
    """
    
    # ChatPromptTemplate.from_messages: 여러 메시지로 구성된 프롬프트 템플릿 생성
    # ("system", system_prompt): 에이전트의 역할 정의 메시지
    # ("user", "{input}"): 사용자의 실제 입력이 들어갈 자리 (변수)
    # AIMessage(content=""): AI의 응답을 위한 빈 메시지 (구조상 필요)
    # ("placeholder", "{agent_scratchpad}"): 도구 사용 기록이 동적으로 삽입될 자리
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt), 
        ("user", "{input}"), 
        AIMessage(content=""), 
        ("placeholder", "{agent_scratchpad}")
    ])
    
    # llm.bind_tools(tools): LLM에게 사용 가능한 도구 목록을 알려줌
    agent = prompt | llm.bind_tools(tools)
    
    def agent_node_func(state: WorkerState):
        """
        Worker 에이전트의 노드 함수
        이 함수는 그래프 실행 시 반복적으로 호출됩니다
        
        동작 과정
        1. 현재 상태(state)에서 메시지들을 분석
        2. 최초 사용자 입력과 이후의 도구 실행 기록을 분리
        3. LLM을 호출하여 다음 행동 결정
        4. LLM 응답을 상태에 추가하여 반환
        """
        # state['messages'][0]: 항상 최초의 사용자 질문 (HumanMessage)
        input_content = state['messages'][0].content
        
        # state['messages'][1:]: 이후의 모든 메시지
        # 여기에는 AI의 도구 호출 결정, 도구 실행 결과(ToolMessage) 등이 포함됨
        agent_scratchpad = state['messages'][1:]
        
        # agent.invoke: LLM을 실행하여 다음 행동 결정
        # 반환값은 AIMessage로, 아래의 두 가지 형태 중 하나
        # 1) 도구 호출 요청 (tool_calls 속성에 정보 포함)
        # 2) 최종 답변 (content에 텍스트 포함)
        response = agent.invoke({
            "input": input_content, 
            "agent_scratchpad": agent_scratchpad
        })
        
        return {"messages": [response]}
        
        
    workflow = StateGraph(WorkerState)
    
    workflow.add_node("agent", agent_node_func)
    
    # ToolNode: LangGraph 내장 노드로, 도구 실행을 자동 처리
    # tools 리스트의 도구들을 실행하고 결과를 ToolMessage로 반환
    workflow.add_node("tools", ToolNode(tools))
    
    workflow.set_entry_point("agent")
    
    # add_conditional_edges: 조건에 따라 다른 노드로 이동하는 엣지 추가
    # 
    # 첫 번째 인자: 출발 노드 ("agent")
    # 
    # 두 번째 인자: 조건 판단 함수 (lambda)
    #   lambda란? 이름 없는 간단한 함수를 만드는 방법
    #   lambda state: "결과" 는 def 함수명(state): return "결과" 와 동일
    #   
    #   여기서 lambda의 역할
    #   - 입력: state (현재 그래프의 상태)
    #   - 처리: state["messages"][-1].tool_calls를 확인
    #            (가장 최근 메시지에 도구 호출 요청이 있는지 체크)
    #   - 출력: "tools" 또는 END (문자열)
    #
    # 세 번째 인자: 조건 결과와 실제 노드를 매핑하는 딕셔너리
    #   - lambda가 "tools"를 반환하면 → "tools" 노드로 이동
    #   - lambda가 END를 반환하면 → 그래프 종료
    workflow.add_conditional_edges(
        "agent", 
        lambda state: "tools" if state["messages"][-1].tool_calls else END,
        {"tools": "tools", END: END}
    )
    
    workflow.add_edge("tools", "agent")
    
    print(f"Worker Agent '{name}' 생성 완료.")
    
    return workflow.compile()

# 각 역할별로 Worker Agent 생성
# 각 에이전트는 독립적인 그래프이지만, 동일한 패턴으로 작동
searcher = create_worker_agent("Searcher", [web_search], "당신은 웹 검색 전문가입니다.")
doc_writer = create_worker_agent("DocWriter", [write_document], "당신은 문서 작성 전문가입니다.")
note_taker = create_worker_agent("NoteTaker", [take_notes], "당신은 정보 요약 전문가입니다.")
chart_generator = create_worker_agent("ChartGenerator", [generate_chart], "당신은 차트 생성 전문가입니다.")

# ============================================================
#  Part 3: Team Supervisor Layer (중간 관리자 계층)
# ============================================================

# Pydantic BaseModel: 데이터 유효성 검사와 구조화된 출력을 위한 클래스
class SubTask(BaseModel):
    """
    하나의 하위 작업을 표현하는 데이터 모델
    LLM이 이 구조에 맞춰 작업을 생성하도록 강제함
    """
    # Field: 각 필드의 설명을 LLM에게 제공 (더 정확한 출력 생성)
    task: str = Field(description="Worker Agent에게 할당할 구체적인 작업 내용")
    assignee: str = Field(description="이 작업을 수행할 Worker Agent의 이름")

class Tasks(BaseModel):
    """
    여러 하위 작업을 담는 컨테이너
    LLM은 이 모델에 맞춰 JSON 형태로 작업 목록을 생성
    """
    tasks: List[SubTask]

class TeamState(TypedDict):
    """
    팀 내부(노드)에서 공유되는 상태 (메모리)
    """
    # 상위 관리자와의 대화 기록 (operator.add로 누적)
    messages: Annotated[List[BaseMessage], operator.add]
    
    # 팀장이 생성한 하위 작업 목록 (누적 아님, 덮어쓰기)
    tasks: List[SubTask]
    
    # 각 팀원의 작업 결과를 담는 딕셔너리
    results: Dict

def create_team_supervisor(name: str, members: Dict[str, any], system_prompt: str):
    """
    Team Supervisor(팀장)를 생성하는 팩토리 함수
    
    매개변수
        name: 팀의 이름
        members: {팀원이름: 팀원에이전트그래프} 형태의 딕셔너리
        system_prompt: 팀장의 역할을 정의하는 프롬프트
    
    반환값
        컴파일된 팀 그래프 (팀장 + 여러 팀원으로 구성)
    """
    
    # members 딕셔너리에서 키(팀원 이름)들만 추출
    member_names = list(members.keys())
    
    def supervisor_node(state: TeamState):
        """
        팀장 노드: 상위 관리자의 지시를 분석하고 팀원들에게 작업 분배
        
        입력: 상위 관리자의 요청 (state["messages"]의 마지막 메시지)
        출력: 하위 작업 목록 (state["tasks"]에 저장)
        """
        print(f"--- TEAM SUPERVISOR ({name}): 작업을 분해하고 할당합니다 ---")
        
        # 프롬프트 구성: 시스템 역할 + 구체적인 작업 분해 요청
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("user", "주어진 요청: {input}\n\n이 요청을 다음 팀원들에게 할당할 구체적인 하위 작업 목록으로 분해해주세요: {members_str}")
        ])
        
        # with_structured_output(Tasks): LLM이 Tasks 모델 형식으로 출력하도록 강제
        # 이를 통해 JSON 파싱 없이 바로 Python 객체로 받을 수 있음
        chain = prompt | llm.with_structured_output(Tasks)
        
        # state["messages"][-1].content: 가장 최근 메시지 (상위 관리자의 지시)
        task_list_obj = chain.invoke({
            "input": state["messages"][-1].content, 
            "members_str": ", ".join(member_names)  # 팀원 이름들을 문자열로 변환
        })
        
        # task_list_obj.tasks: Tasks 객체의 tasks 필드 (SubTask 리스트)
        tasks = task_list_obj.tasks
        
        print("\n   [생성된 하위 작업 목록]")
        for task in tasks:
            print(f"   - 담당자: {task.assignee}, 작업: {task.task}")
        print()
        
        # 생성된 작업 목록을 상태에 저장, results는 빈 딕셔너리로 초기화
        return {"tasks": tasks, "results": {}}
    
    def worker_node(state: TeamState, member_name: str, member_agent: any):
        """
        팀원 노드: 자신에게 할당된 작업을 찾아서 실행
        
        매개변수
            state: 현재 팀 상태
            member_name: 이 노드를 실행하는 팀원의 이름
            member_agent: 이 팀원의 Worker Agent 그래프 (컴파일된 그래프 객체)
        
        동작
            1. tasks에서 자신에게 할당된 작업 찾기
            2. Worker Agent 그래프를 invoke하여 작업 수행
            3. 결과를 results 딕셔너리에 추가
        """
        print(f"--- WORKER ({member_name}): 작업을 수행합니다 ---")
        
        # next(): 이터레이터에서 조건에 맞는 첫 번째 요소 반환
        # (t for t in state["tasks"] if t.assignee == member_name): 제너레이터 표현식
        #   - state["tasks"]의 각 작업(t)을 순회하며 자신에게 할당된 작업 찾기
        # None: 조건에 맞는 작업이 없을 때 반환할 기본값
        task = next((t for t in state["tasks"] if t.assignee == member_name), None)
        
        # 할당된 작업이 없으면 빈 딕셔너리 반환 (상태 변경 없음)
        if task is None: 
            return {}
        
        print(f"   -> 실행 작업: '{task.task}'")
        
        # member_agent.invoke(): Worker Agent 그래프 실행 (그래프 중첩부)
        # 입력: {"messages": [HumanMessage(...)]} 형태
        # 출력: {"messages": [..., AIMessage(최종답변)]} 형태
        # 즉, 하나의 그래프가 다른 그래프를 호출하는 구조
        result = member_agent.invoke({"messages": [HumanMessage(content=task.task)]})
        
        # result["messages"][-1].content: Worker의 최종 답변 추출
        result_content = result["messages"][-1].content
        
        print(f"   -> 결과 (일부): {result_content[:150]}...")
        
        # **state["results"]: 기존 results 딕셔너리를 언팩
        # member_name: result_content: 새 결과 추가
        # {**기존, 새키:새값}: 딕셔너리 병합 패턴
        return {"results": {**state["results"], member_name: result_content}}

    def synthesizer_node(state: TeamState):
        """
        결과 종합 노드: 모든 팀원의 결과를 모아 최종 보고서 작성
        
        입력: state["results"] (팀원별 작업 결과 딕셔너리)
        출력: 최종 보고서 (state["messages"]에 AIMessage로 추가)
        """
        print(f"--- TEAM SUPERVISOR ({name}): 팀원들의 결과를 종합합니다 ---")
        
        # results 딕셔너리를 문자열로 포맷팅
        # "\n---\n".join(...): 각 결과를 "---"로 구분
        # f"[{k}의 결과]\n{v}": 각 팀원의 이름과 결과를 포맷팅
        # for k, v in state["results"].items(): 딕셔너리의 키-값 쌍 순회
        results_str = "\n---\n".join(
            f"[{k}의 결과]\n{v}" for k, v in state["results"].items()
        )
        
        print("\n   [종합할 결과물들]")
        print(results_str)
        print()
        
        # 결과 종합 프롬프트: 여러 팀원의 결과를 하나의 보고서로 통합
        prompt = ChatPromptTemplate.from_template(
            "다음은 팀원들의 작업 결과입니다. 이를 종합하여 사용자의 원래 요청에 대한 최종 보고서를 작성해주세요.\n\n{results_str}"
        )
        
        # LLM 체인 실행: 프롬프트 → LLM → 문자열 파싱
        final_report = (prompt | llm | StrOutputParser()).invoke({"results_str": results_str})
        
        # 최종 보고서를 AIMessage로 감싸 messages에 추가
        # 이 메시지가 상위 관리자에게 전달됨
        return {"messages": [AIMessage(content=final_report)]}
    
    # 팀 워크플로우 그래프 구성 시작
    workflow = StateGraph(TeamState)
    
    # 팀장 노드 추가 (작업 분해 및 할당)
    workflow.add_node(f"{name}_supervisor", supervisor_node)
    
    # 각 팀원에 대해 반복:
    for member_name, member_agent in members.items():
        # lambda 함수 내부의 기본 인자 패턴: member_name=member_name
        #
        # 반복문에서의 lambda 사용 시 주의점
        # - 잘못된 방법: lambda state: worker_node(state, member_name, member_agent)
        #   → 반복문이 끝나면 member_name은 마지막 팀원 이름으로 고정됨
        #   → 모든 노드가 마지막 팀원만 호출 (버그 발생)
        #
        # - 올바른 방법: lambda state, member_name=member_name, ...
        #   → member_name=member_name의 의미
        #     왼쪽 member_name: lambda 함수의 매개변수 (새로운 이름)
        #     오른쪽 member_name: 현재 반복문의 변수 (현재 값을 가져와서 고정)
        #   → 각 노드가 자기 팀원을 정확히 호출 (올바른 동작)
        #
        # Python의 특성
        #   반복문 안에서 lambda를 만들 때, 변수를 그냥 쓰면 참조만 하고
        #   기본 인자로 쓰면 현재 값을 복사해서 저장함
        workflow.add_node(
            member_name, 
            lambda state, member_name=member_name, member_agent=member_agent: 
                worker_node(state, member_name, member_agent)
        )
        
        # 팀장 → 각 팀원으로 엣지 추가 (병렬 실행)
        # 팀장 노드가 끝나면 모든 팀원 노드가 동시에 실행됨
        workflow.add_edge(f"{name}_supervisor", member_name)

    # 결과 종합 노드 추가
    workflow.add_node(f"{name}_synthesizer", synthesizer_node)
    
    # 각 팀원 → 결과 종합 노드로 엣지 추가
    # 모든 팀원의 작업이 완료되어야 결과 종합 노드가 실행됨 (암묵적 병렬 대기)
    for member_name in members.keys():
        workflow.add_edge(member_name, f"{name}_synthesizer")
    
    # 팀 워크플로우의 시작점 설정
    workflow.set_entry_point(f"{name}_supervisor")
    
    # 결과 종합이 끝나면 팀 그래프 종료
    workflow.add_edge(f"{name}_synthesizer", END)
    
    print(f"Team Supervisor '{name}' 그래프 생성 완료.")
    return workflow.compile()

# 전문 팀 생성
research_team = create_team_supervisor(
    "ResearchTeam", 
    {"Searcher": searcher},
    "당신은 리서치 팀장입니다. 사용자의 요청을 분석하여 Searcher에게 웹 검색 작업을 지시하세요."
)

documentation_team = create_team_supervisor(
    "DocumentationTeam", 
    {
        "NoteTaker": note_taker, 
        "DocWriter": doc_writer, 
        "ChartGenerator": chart_generator
    },
    "당신은 문서 작성 팀장입니다. 사용자의 요청을 분석하여 NoteTaker, DocWriter, ChartGenerator에게 작업을 분배하세요."
)

# ============================================================
#  Part 4: Top Supervisor Layer (최상위 지휘자 계층)
# ============================================================

class TopSupervisorState(TypedDict): 
    """
    최상위 관리자의 상태
    전체 대화 기록만 관리 (작업 분배는 하위 팀에 위임)
    """
    messages: Annotated[List[BaseMessage], operator.add]

class Route(BaseModel):
    """
    최상위 관리자의 라우팅 결정을 표현하는 모델
    LLM이 어느 팀으로 작업을 보낼지 구조화된 형태로 결정
    """
    next_node: str = Field(
        description="'ResearchTeam', 'DocumentationTeam' 또는 'FinalAnswer' 중 하나여야 함"
    )

def top_supervisor_router(state: TopSupervisorState):
    """
    최상위 관리자 라우터: 사용자 요청을 분석하여 적절한 팀 선택
    
    동작
        1. 사용자의 요청 분석
        2. 요청의 성격 파악 (정보 조사? 문서 작성? 단순 대화?)
        3. 적절한 팀 이름 또는 'FinalAnswer' 반환
    
    반환값
        문자열 ("ResearchTeam", "DocumentationTeam", 또는 "FinalAnswer")
    """
    print("--- TOP SUPERVISOR: 작업을 분석하고 팀을 선택합니다 ---")
    
    # 프롬프트 구성: 시스템 메시지 + 현재까지의 대화 기록
    # state["messages"]: 사용자의 최신 요청을 포함한 전체 대화
    prompt = ChatPromptTemplate.from_messages([
        ("system", 
         "당신은 AI 에이전트들의 최상위 지휘자입니다. "
         "사용자의 요청을 분석하여 처리할 팀을 선택하세요. "
         "'ResearchTeam'은 정보 조사를, 'DocumentationTeam'은 문서 작성을 담당합니다. "
         "간단한 대화는 'FinalAnswer'로 직접 답변하세요.")
    ] + state["messages"])
    
    # with_structured_output(Route): LLM이 Route 모델 형식으로 결정 반환
    # decision.next_node: "ResearchTeam", "DocumentationTeam", 또는 "FinalAnswer"
    decision = (prompt | llm.with_structured_output(Route)).invoke({})
    
    print(f"   -> Top Supervisor 결정: {decision.next_node}")
    
    # 결정된 팀 이름을 문자열로 반환
    return decision.next_node

# 최상위 워크플로우 그래프 생성
workflow = StateGraph(TopSupervisorState)

# 각 팀을 하나의 노드로 추가 (그래프 중첩부)
#
# Lambda의 역할: 양방향 변환기 (상위 <-> 하위)
#
# 양방향 변환 과정
#
#   [1단계: 상위 → 하위] 입력 전달
#   lambda state:  ← state는 TopSupervisorState 형식
#       research_team.invoke(state)  ← state를 그대로 하위 팀에 전달
#                                     (TeamState도 messages 필드가 있어서 호환됨)
#
#   [2단계: 하위 팀 내부 실행]
#       팀장 → 팀원들 → 결과종합
#       최종 출력: {"messages": [..., AIMessage(최종보고서)]}
#
#   [3단계: 하위 → 상위] 결과 추출 및 변환
#       ["messages"][-1]  ← 팀의 전체 결과에서 최종 보고서만 추출
#       {"messages": [...]}  ← 상위 그래프가 받을 수 있는 형식으로 재포장
#
# 핵심 내용
#   lambda는 단순히 전달만 하는 게 아니라
#   1) 상위 → 하위: state를 하위 그래프에 전달 (입력 변환)
#   2) 하위 → 상위: 결과를 상위 형식으로 변환 (출력 변환)
#   양방향 모두 처리

workflow.add_node(
    "ResearchTeam", 
    lambda state: {"messages": [research_team.invoke(state)["messages"][-1]]}
)

workflow.add_node(
    "DocumentationTeam", 
    lambda state: {"messages": [documentation_team.invoke(state)["messages"][-1]]}
)

# FinalAnswer 노드: 단순 대화는 최상위 LLM이 직접 답변
# 
# 여기서 lambda의 역할: LLM 호출 래퍼(포장 함수)
#   LLM의 응답을 그래프 상태 형식으로 변환
#
# lambda state: {...}
#   - 입력: state (대화 기록 포함)
#   - 처리: llm.invoke(state["messages"]) 
#           → 전체 대화 기록을 LLM에 전달하여 답변 생성
#   - 출력: {"messages": [AIMessage(답변)]} 형식으로 포장
#
# lambda를 쓰는 이유
#   - add_node는 함수를 받아야 함
#   - llm.invoke는 그래프 상태가 아닌 메시지 리스트를 받음
#   - lambda로 형식을 맞춰주는 것 (어댑터 패턴)

workflow.add_node(
    "FinalAnswer", 
    lambda state: {"messages": [llm.invoke(state["messages"])]}
)

# 조건부 시작: START → top_supervisor_router의 결정에 따라 분기
# add_conditional_edges의 세 번째 인자: 라우터 반환값 → 실제 노드 매핑
# 예: 라우터가 "ResearchTeam" 반환 → "ResearchTeam" 노드 실행

workflow.add_conditional_edges(
    START,  # 그래프 시작 지점
    top_supervisor_router,  # 조건 판단 함수
    {
        "ResearchTeam": "ResearchTeam", 
        "DocumentationTeam": "DocumentationTeam", 
        "FinalAnswer": "FinalAnswer"
    }
)

# 각 팀/노드 실행 후 그래프 종료
workflow.add_edge("ResearchTeam", END)
workflow.add_edge("DocumentationTeam", END)
workflow.add_edge("FinalAnswer", END)

# 전체 시스템을 하나의 실행 가능한 객체로 컴파일
# 이 시점에서 3단계 계층 (Top Supervisor → Team → Worker) 모두 통합됨
app = workflow.compile()

print("\n최상위 Supervisor 그래프 컴파일 및 전체 시스템 결합 완료!")

# 그래프 시각화 시도
try:
    # xray=True: 중첩된 모든 서브그래프를 펼쳐서 하나의 이미지로 표시
    # xray=False이면 팀을 블랙박스로 표시 (내부 구조 숨김)
    img_bytes = app.get_graph(xray=True).draw_mermaid_png()
    with open("hierarchical_agent_graph.png", "wb") as f: 
        f.write(img_bytes)
    print("전체 계층 구조를 'hierarchical_agent_graph.png' 파일로 저장했습니다.")
except Exception as e:
    print(f"그래프 시각화 실패 (Graphviz 필요): {e}")

# ============================================================
#  Part 5: 시스템 실행
# ============================================================

async def main():
    """
    시스템의 메인 실행 루프
    
    동작 과정
        1. 사용자 입력 받기
        2. 대화 기록에 추가
        3. 전체 그래프 실행 (Top Supervisor → 팀 선택 → 팀 실행)
        4. 최종 답변 출력
        5. 답변을 대화 기록에 추가 (다음 턴의 컨텍스트로 사용)
    
    무한 루프로 멀티턴 대화 지원
    """
    print("\n" + "=" * 80)
    print("계층적 에이전트 팀 시스템을 시작합니다.")
    print("=" * 80)
    
    # 대화 기록을 저장할 리스트 초기화
    # 이 리스트는 루프를 돌면서 계속 누적되어 대화의 맥락을 유지
    chat_history = []
    
    # 무한 루프: 사용자가 'quit', 'exit', '종료'를 입력할 때까지 반복
    while True:
        # input(): 사용자로부터 텍스트 입력 받기
        # .strip(): 앞뒤 공백 제거
        user_question = input("\nYou: ").strip()
        
        # 종료 명령어 체크 (대소문자 무시)
        if user_question.lower() in ['quit', 'exit', '종료']: 
            break
        
        # 빈 입력이면 다시 입력 받기
        if not user_question: 
            continue
        
        # HumanMessage: 사용자의 메시지를 나타내는 LangChain 메시지 객체
        # content: 실제 메시지 내용
        chat_history.append(HumanMessage(content=user_question))
        
        # 그래프 실행을 위한 입력 준비
        # TopSupervisorState 형식에 맞춰 messages 키로 전달
        inputs = {"messages": chat_history}
        
        try:
            # 사용자에게 AI가 응답 중임을 표시
            # end="", flush=True: 줄바꿈 없이 즉시 출력
            print("\nAI: \n", end="", flush=True)
            
            # await app.ainvoke(inputs): 전체 그래프를 비동기로 실행
            # ainvoke는 invoke의 비동기 버전 (async/await 사용)
            # 
            # 실행 흐름
            # 1. START → top_supervisor_router 호출
            # 2. 라우터가 팀 선택 (예: "ResearchTeam")
            # 3. 선택된 팀의 그래프 전체 실행
            #    - 팀장: 작업 분해 및 할당
            #    - 팀원들: 병렬로 각자 작업 수행 (Worker 그래프 invoke)
            #    - 결과 종합: 최종 보고서 생성
            # 4. 팀의 최종 보고서가 상위 그래프로 반환
            # 5. END → 그래프 실행 종료
            #
            # 반환값: final_state는 TopSupervisorState 형태
            # {"messages": [HumanMessage(...), AIMessage(팀의최종보고서)]}
            final_state = await app.ainvoke(inputs)
            
            # final_state['messages'][-1]: 마지막 메시지 (AIMessage)
            # .content: 메시지의 실제 텍스트 내용 (팀의 최종 보고서)
            final_answer = final_state['messages'][-1].content
            
            # 최종 답변 출력
            print(final_answer, end="", flush=True)
            
            # AIMessage: AI의 응답을 나타내는 메시지 객체
            # 대화 기록에 추가하여 다음 턴에서 참조 가능하도록 함
            # 예: 사용자가 "그럼 이전 검색 결과를 요약해줘"라고 하면
            #     이 기록을 통해 이전 답변을 참조할 수 있음
            chat_history.append(AIMessage(content=final_answer))
            
            print("\n" + "-" * 80)
            
        except Exception as e:
            print(f"\n[오류 발생] {e}")


if __name__ == "__main__":
    asyncio.run(main())